# TypeScript through a Python lens

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 37 §37.1 · type: concept-lab

**The promise:** by the end you can read and reason about TypeScript — structural typing, discriminated unions + narrowing, generics, and *why types vanish at runtime* — by mapping each idea to its Python equivalent.

**The medium can't run React.** A Python kernel can't execute TypeScript or Next.js, so this is a *concept-lab*, not an app build. Every TS idea here is made runnable by simulating it in Python; the TS appears in markdown for reading. The real frontend code lives in [`templates/web-starter/`](../../../templates/web-starter/) and [`capstone/web/`](../../../capstone/web/), and this notebook ends by pointing there.

Runs free and offline by default (`MOCK=1`): pure standard-library Python, no Node/Deno, no API key.

## 🧠 Why this matters

You already hold most of TypeScript's concepts — you write Python type hints every day. The job here is *translation*, not learning a new discipline: a modern frontend is a typed application with the same shape as your FastAPI backend (a data layer, a domain layer, a presentation layer). Find the types, find the data flow, find the boundaries, and the unfamiliar syntax becomes a dialect.

The one idea that genuinely surprises Python engineers: TypeScript's types are checked at *compile time* and then **erased entirely** — the browser runs plain JavaScript and has never heard of your interfaces. Python's hints are advisory at runtime *unless* something like Pydantic enforces them; TypeScript's are *gone* at runtime, full stop. That single fact drives the chapter's biggest pitfall, so we'll feel it directly. See §37.1.

## Objectives & prereqs

**By the end you can:**
- Map TS concepts to their Python equivalents (`interface`→`Protocol`, literal union→`Literal`, generics→`TypeVar`).
- Explain *structural* typing and show that any object with the right shape satisfies an interface.
- Model an `AgentEvent` discriminated union and *narrow* it the way a `switch` does in TS — the backbone of streamed-event UIs.
- Spot the "types validate nothing at runtime" trap and fix it by validating at the boundary (Zod ↔ Pydantic).

**Prereqs:** Ch 4 (typing/async mental model). Chapter 37 §37.1 read. A Deno or `tslab` kernel is **optional** — every TS idea has a pure-Python fallback, so the Python kernel alone runs the whole notebook.

**Packages:** standard library only (`typing`, `dataclasses`). Optional `pydantic` (in `requirements.txt`) anchors the boundary-validation analogy; the notebook degrades gracefully if it's absent.

In [ ]:
# Setup — imports, env, the MOCK switch, a fixed seed, and a kernel note.
import os
import random

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default): fully offline, pure-Python simulation of TS ideas. No model calls
# happen anywhere in this notebook — the switch exists for consistency with the rest of
# the course and to gate the (entirely optional) live-API exercise.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(37)  # determinism for anything stochastic

# A real TypeScript kernel (Deno / tslab) is OPTIONAL. We never require it: every TS
# snippet below lives in a markdown cell for reading, and its behavior is reproduced in
# Python so the notebook runs top-to-bottom with the plain Python kernel.
print(f"MOCK mode: {MOCK}  (offline, pure-Python simulation of TypeScript ideas)")
print("TS kernel: not required — Python fallbacks cover every cell.")

## The concept map: TS ↔ Python

Before any code, anchor the dialect. Almost everything you'll read in a `.tsx` file has a Python concept you already know:

| TypeScript | Python | What it's for |
|---|---|---|
| `interface RunEvent { ... }` | `typing.Protocol` | a *shape* an object must match (structural) |
| `"token" \| "done"` (literal union) | `Literal["token", "done"]` | a fixed set of allowed values |
| discriminated union + `switch` | tagged classes + `match` | model variants, then narrow to one |
| `payload?: string` | `str \| None` | optional field |
| `function first<T>(xs: T[])` | `def first(xs: list[T])` (`TypeVar`) | generics |
| `schema.parse(json)` (**Zod**) | `Model.model_validate(json)` (**Pydantic**) | validate at the boundary |
| `const` / `let` (never `var`) | (no direct analog; just names) | declarations |
| `===` | `==` (Python `==` is value equality) | the equality you mean |
| `(x) => x * 2` | `lambda x: x * 2` | arrow function |
| `async` / `await` | `async` / `await` | *all* JS I/O is async (Ch 4) |

Read the table once; we'll make the load-bearing rows runnable below.

## Structural typing: shape is the contract

TypeScript typing is *structural*, not nominal: any object with the right shape satisfies an `interface` — no inheritance, no registration. It's duck typing that the compiler verifies, closer to Python's `Protocol` than to its classes. Here's the book's `RunEvent` shape from §37.1:

```ts
// TypeScript — for reading. Types describe shapes; checking happens at compile time only.
interface RunEvent {
  runId: string;
  step: number;
  kind: "tool_call" | "token" | "done";   // literal union — like Literal[]
  payload?: string;                        // optional, like str | None
}

function describe(e: RunEvent): string {
  return `run ${e.runId}, step ${e.step}: ${e.kind}`;
}
```

The Python mirror below uses `Protocol`. Notice that `describe` never asks "are you an instance of RunEvent?" — it only asks "do you have these attributes?" A `dataclass` that was never told about the protocol still satisfies it.

In [ ]:
from dataclasses import dataclass
from typing import Literal, Optional, Protocol, runtime_checkable


@runtime_checkable
class RunEvent(Protocol):
    """Python mirror of the TS `interface RunEvent` — a *shape*, matched structurally."""
    run_id: str
    step: int
    kind: Literal["tool_call", "token", "done"]
    payload: Optional[str]


def describe(e: RunEvent) -> str:
    return f"run {e.run_id}, step {e.step}: {e.kind}"


# This dataclass was NEVER told it implements RunEvent — it just has the right shape.
@dataclass
class WireEvent:
    run_id: str
    step: int
    kind: str
    payload: Optional[str] = None


evt = WireEvent(run_id="r-001", step=3, kind="token", payload="hel")
print(describe(evt))                       # works: shape matches, no inheritance needed
print("satisfies RunEvent?", isinstance(evt, RunEvent))  # structural check at runtime

**What you just saw.** `WireEvent` has no relationship to `RunEvent` in its class hierarchy — yet it satisfies it, because the *shape* matches. That's structural typing: the contract is the set of fields, not a name in an inheritance tree. TypeScript checks this at compile time; Python's `@runtime_checkable` `Protocol` lets us check it at runtime for the demo.

## Discriminated unions: the backbone of streamed agent UIs

Here's the single most useful TS pattern for AI frontends. A **discriminated union** is a union of object types that share a literal *tag* field (here, `kind`). Streamed agent events are exactly that — each chunk off the wire is a `token`, a `tool_call`, or a `done`. This foreshadows Chapter 38, where these events drive a live UI.

```ts
// TypeScript — for reading. One value, three possible shapes, tagged by `kind`.
type AgentEvent =
  | { kind: "token"; text: string }
  | { kind: "tool_call"; tool: string; args: unknown }
  | { kind: "done"; answer: string };

function render(e: AgentEvent) {
  switch (e.kind) {
    case "token":     return e.text;          // narrowed: only `text` exists here
    case "tool_call": return `[${e.tool}]`;   // narrowed: `tool` exists here
    case "done":      return e.answer;
  }
}
```

Inside each `case`, the compiler *narrows* the union to one variant and lets you touch only that variant's fields — pattern matching by another name. The Python equivalent is tagged classes + `match`/`case`, which narrows the same way.

In [ ]:
from dataclasses import dataclass
from typing import Union


# Each variant carries the literal tag `kind` plus the fields that ONLY it has.
@dataclass
class Token:
    text: str
    kind: Literal["token"] = "token"


@dataclass
class ToolCall:
    tool: str
    args: object
    kind: Literal["tool_call"] = "tool_call"


@dataclass
class Done:
    answer: str
    kind: Literal["done"] = "done"


AgentEvent = Union[Token, ToolCall, Done]


def render(e: AgentEvent) -> str:
    """Python mirror of the TS `switch (e.kind)` — match narrows to one variant."""
    match e:
        case Token(text=text):       # narrowed: `text` exists here
            return text
        case ToolCall(tool=tool):    # narrowed: `tool` exists here
            return f"[{tool}]"
        case Done(answer=answer):
            return answer
    raise ValueError(f"unhandled event: {e!r}")  # TS would flag this at compile time


stream = [Token("Hel"), Token("lo"), ToolCall("search", {"q": "weather"}), Done("Sunny.")]
for ev in stream:
    print(f"{ev.kind:<10} -> {render(ev)!r}")

## 🔮 Predict: which case can touch which field?

In the TS `switch` above (and the Python `match` we just ran), the compiler narrows the union inside each branch. Predict, *before running the next cell*:

- In the `"token"` branch, is `e.text` reachable? Is `e.answer`?
- In the `"done"` branch, is `e.answer` reachable? Is `e.text`?

Write down your answer. Then run the cell, which probes each field on each variant and reports which are present — the runtime shadow of what the type-checker enforces statically.

In [ ]:
# Probe: for each variant, which of {text, tool, answer} actually exist?
# In TypeScript this is enforced at COMPILE time (e.text in the `done` branch won't
# compile). Here we observe the same truth at runtime via the present fields.
variants = {
    "token":     Token("Hi"),
    "tool_call": ToolCall("search", {"q": "x"}),
    "done":      Done("Final answer."),
}
fields = ["text", "tool", "answer"]

print(f"{'kind':<11}" + "".join(f"{f:<9}" for f in fields))
for kind, ev in variants.items():
    marks = ["yes" if hasattr(ev, f) else "—" for f in fields]
    print(f"{kind:<11}" + "".join(f"{m:<9}" for m in marks))

print("\nSo: the 'token' branch may touch .text but NOT .answer;")
print("the 'done' branch may touch .answer but NOT .text. Narrowing = safety per branch.")

**What you just saw.** Each variant exposes only its own fields. The discriminant (`kind`) tells both the compiler and you which shape you're holding, so a `done` event can never accidentally read `.text`. This is *why* discriminated unions are the right model for streamed events: every chunk is unambiguously one shape, and the renderer handles each exhaustively. TypeScript even errors if you forget a `case` (exhaustiveness); our Python `raise` at the end is the runtime stand-in for that guarantee.

## Generics: read far more than you write

Generics in TS work like Python's `TypeVar`, just terser. `function first<T>(xs: T[]): T | undefined` is `def first(xs: list[T]) -> T | None`. The angle brackets mean "this type is a parameter." You'll *read* generics constantly and *write* them rarely — recognize the shape and move on.

In [ ]:
from typing import Optional, TypeVar

T = TypeVar("T")


def first(xs: list[T]) -> Optional[T]:
    """TS: function first<T>(xs: T[]): T | undefined.

    The element type T flows through: give it strings, you get a string back; give it
    AgentEvents, you get an AgentEvent. One definition, many concrete types.
    """
    return xs[0] if xs else None


print("first of strings :", first(["a", "b", "c"]))
print("first of events  :", first(stream))   # returns the first AgentEvent
print("first of empty   :", first([]))       # None  (TS: undefined)

## ⚠️ Pitfall: TS types are *erased* — they validate nothing

The trap for Pydantic-trained engineers: TypeScript types **vanish at runtime**. Declaring that a `fetch` response is `RunEvent` is a promise to the *compiler*, not a check on the *data*. A malformed API response sails straight through the (now-erased) type and explodes three components later, far from the cause.

```ts
// TypeScript — this COMPILES and then LIES. No runtime check happens.
const res = await fetch("/api/run");
const event = (await res.json()) as RunEvent;  // a cast: "trust me, compiler"
// If the server actually returned { kind: "oops" }, `event.kind` is now a lie.
```

The fix mirrors your backend habit: **validate at the boundary**. The *Zod* library is the ecosystem's Pydantic — define a schema, call `schema.parse(json)`, and you get a runtime error at the edge *plus* an inferred static type for free.

```ts
// TypeScript with Zod — validates at runtime AND infers the static type.
import { z } from "zod";
const RunEventSchema = z.object({
  runId: z.string(),
  step: z.number(),
  kind: z.enum(["tool_call", "token", "done"]),
  payload: z.string().optional(),
});
const event = RunEventSchema.parse(await res.json());  // throws on bad data, at the edge
```

The cell below makes the difference *felt* in Python: a bare cast accepts garbage; a Pydantic `model_validate` (Zod's twin) rejects it at the boundary.

In [ ]:
# A malformed payload the server should never have sent (kind is not in the union).
bad_payload = {"runId": "r-002", "step": "three", "kind": "oops"}

# 1) The TS-cast equivalent: trust the shape, check nothing. The lie propagates.
casted = bad_payload  # `as RunEvent` does exactly this much at runtime: nothing.
print("cast (no validation): accepted blindly ->", casted)
print("   ...the 'step' is the string 'three' and 'kind' is 'oops'; nobody noticed yet.\n")

# 2) The Zod equivalent: validate at the boundary with Pydantic. Catches it here, not
#    three components later. Degrades gracefully if pydantic isn't installed.
try:
    from pydantic import BaseModel, ValidationError

    class RunEventModel(BaseModel):           # Zod's z.object(...) twin
        runId: str
        step: int
        kind: Literal["tool_call", "token", "done"]
        payload: Optional[str] = None

    try:
        RunEventModel.model_validate(bad_payload)
        print("validate: unexpectedly accepted (should not happen)")
    except ValidationError as e:
        print("validate (Zod/Pydantic): REJECTED at the boundary — the errors:")
        for err in e.errors():
            print(f"   {'.'.join(map(str, err['loc']))}: {err['msg']}")
except ModuleNotFoundError:
    print("validate: pydantic not installed — pure-Python fallback check:")
    allowed = {"tool_call", "token", "done"}
    errs = []
    if not isinstance(bad_payload.get("step"), int):
        errs.append("step: expected int")
    if bad_payload.get("kind") not in allowed:
        errs.append(f"kind: {bad_payload.get('kind')!r} not in {sorted(allowed)}")
    print("   REJECTED at the boundary —", "; ".join(errs))

**What you just saw.** The cast accepted a payload with a non-integer `step` and an out-of-union `kind` — a time bomb. The boundary validator caught both *at the edge*, with a precise error, exactly where the bad data entered. In TS, `as RunEvent` is the bomb; `Zod.parse(...)` is the validator. Type every boundary twice: once for the compiler, once for reality.

## 🎯 Senior lens

A senior never lets a static type stand in for a runtime check at a *boundary* — the network, `localStorage`, `process.env`, anything the compiler can't see. Inside the program, lean on the types and trust them; *at the edge*, validate with Zod (or Pydantic, server-side) and let the inferred type flow inward from the validated value. The discriminated union earns its keep here too: the moment you parse an event at the boundary, its `kind` tag makes every downstream `switch` exhaustive and safe. The bug a senior is paid to prevent is the one where a confidently-typed `fetch` result was never the shape it claimed, and the crash surfaces three components away from the lie.

## Recap

- TypeScript is JavaScript + a static type system; most concepts map straight onto Python's typing tools.
- Typing is **structural** — the shape is the contract (`interface` ≈ `Protocol`), no inheritance needed.
- **Discriminated unions** (a literal `kind` tag + per-variant fields) model streamed agent events; a `switch`/`match` **narrows** to one variant so each branch is type-safe. This is the backbone of Chapter 38's streaming UI.
- **Generics** = `TypeVar`; you read them far more than you write them.
- The big trap: TS types are **erased at runtime and validate nothing**. Validate at the boundary with **Zod** (the ecosystem's Pydantic). Type every boundary twice.

## Exercises

Each task *changes* something; predict the outcome before you run it.

1. **Add a variant.** Extend `AgentEvent` with an `Error` variant `{kind: "error"; message: str}`. Predict what happens to `render` *before* you add a `case` for it (hint: the `raise` at the end fires). Then handle it.
2. **Exhaustiveness.** Remove the `Done` case from `render`'s `match` and feed it a `Done` event. Predict the failure mode, then observe it — this is the runtime echo of the compile-time error TS would have shown.
3. **Tighten the schema.** In the boundary cell, add a rule that `step` must be `>= 0` (Pydantic: `Field(ge=0)`; fallback: a manual check). Feed it `step=-1` and predict whether the cast path or the validate path catches it.
4. **(Optional, live)** This notebook needs no API key. If you want, set `COMPANION_MOCK=0` and adapt the boundary cell to validate a real JSON response you fetch with `httpx` — the point is identical: validate where the data enters.

In [ ]:
# Exercise scratch space — your code here.


In [ ]:
# Exercise scratch space — your code here.


## Next

- **Next notebook:** [`37-02-react-and-nextjs-mental-model.ipynb`](37-02-react-and-nextjs-mental-model.ipynb) — *the UI is a function of state*, simulated in Python, plus the Next.js server/client split.
- **Book:** §37.1 (TypeScript for backend engineers); the `AgentEvent` union returns in Chapter 38 to drive the live streaming interface.
- **The real code:** the actual TypeScript app you'll direct and review lives in [`templates/web-starter/`](../../../templates/web-starter/) (the Next.js App Router starter — TypeScript · Tailwind · shadcn/ui) and the complete reference frontend in [`capstone/web/`](../../../capstone/web/). You built the mental model here; that's where you apply it.